<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/02e-compiling-and-optimizing-pytorch-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compiling and Optimizing PyTorch Models



In [1]:
%%bash

pip install --upgrade torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 12.6 MB/s eta 0:00:00


In [2]:
import pathlib


import pandas as pd
import torch
from torch import nn, optim, utils
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T


# default linewidth is 80 characters
torch.set_printoptions(linewidth=120)


## Verifying availability of GPU(s)

In [3]:
print(torch.__version__)

2.8.0+cu126


In [4]:
%%bash

nvidia-smi

Wed Nov 12 07:31:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
print(torch.cuda.is_available())

True


In [6]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [7]:
print(DEVICE)

cuda


## Loading the data

In [8]:
DATA_DIR = pathlib.Path("./sample_data")


to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])


train_val_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=True,
                   download=True,
                   transform=to_tensor
               )
)

test_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=False,
                   download=True,
                   transform=to_tensor
               )
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 303kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.57MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.6MB/s]


In [9]:
%%bash

ls ./sample_data/FashionMNIST/raw

t10k-images-idx3-ubyte
t10k-images-idx3-ubyte.gz
t10k-labels-idx1-ubyte
t10k-labels-idx1-ubyte.gz
train-images-idx3-ubyte
train-images-idx3-ubyte.gz
train-labels-idx1-ubyte
train-labels-idx1-ubyte.gz


## Preparing the data

### Train/Val split

In [11]:
_ = torch.manual_seed(42)

train_dataset, val_dataset = (
    utils.data
         .random_split(
             train_val_dataset,
             [55_000, 5_000]
         )
)

### Create the DataLoaders

In [12]:
data_loader_kwargs = {
    "batch_size": 32,
    "num_workers": 2,            # load data in parallel using multiple workers
    "persistent_workers": True,  # keep workers around between epochs
    "pin_memory": True,          # avoid extra copy of data batches
    "prefetch_factor": 2,        # fetch multiple data batches in advance
}


train_data_loader = (
    utils.data
         .DataLoader(
             train_dataset,
             shuffle=True,
             **data_loader_kwargs
         )
)

val_data_loader = (
    utils.data
         .DataLoader(
             val_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

test_data_loader = (
    utils.data
         .DataLoader(
             test_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

## Defining the training and evaluation loop

In [13]:
def evaluate(model_fn, data_loader, metric):
    model_fn.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end


def train(
    model_fn,
    criterion,
    optimizer,
    metric,
    train_data_loader,
    val_data_loader,
    n_epochs,
    log_epochs=1,
    ):

    history = {
        "train_losses": [],
        "val_losses": [],
        "train_metrics": [],
        "val_metrics": [],
    }

    for epoch in range(n_epochs):
        total_train_loss = 0.0
        metric.reset()
        for i, (X_batch, y_batch) in enumerate(train_data_loader):
            model_fn.train()

            # move batches to device
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            # forward pass
            y_pred = model_fn(X_batch)
            train_loss = criterion(y_pred, y_batch)
            total_train_loss += train_loss.item()

            # backward pass
            train_loss.backward()

            # gradient descent step
            optimizer.step()
            optimizer.zero_grad()

            # update our metric
            metric.update(y_pred, y_batch)

        # comute the average (across batches!) training loss
        average_train_loss = total_train_loss / len(train_data_loader)
        history["train_losses"].append(average_train_loss)

        # compute the average (across batched!) validation loss
        with torch.no_grad():
            model_fn.eval()
            total_val_loss = 0.0
            for X_batch, y_batch in val_data_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                y_pred = model_fn(X_batch)
                val_loss = criterion(y_pred, y_batch)
                total_val_loss += val_loss.item()
            average_val_loss = total_val_loss / len(val_data_loader)
            history["val_losses"].append(average_val_loss)

        # compute the training metric after each epoch
        average_train_metric = (
            metric.compute()
                  .item()
        )
        history["train_metrics"].append(average_train_metric)

        # compute the validation metric after each epoch
        average_val_metric = (
            evaluate(
              model_fn,
              val_data_loader,
              metric,
            ).item()
        )
        history["val_metrics"].append(average_val_metric)

        if (epoch + 1) % log_epochs == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}, "
                  f"train loss: {history['train_losses'][-1]:.4f}, "
                  f"val loss: {history['val_losses'][-1]:.4f}, "
                  f"train metric: {history['train_metrics'][-1]:.4f}, "
                  f"val metric: {history['val_metrics'][-1]:.4f}"
            )

    return history


## Putting everything together!

In [14]:
class MLPClassifier(nn.Module):

    def __init__(self, input_size, hidden_layer_sizes, n_classes):
        super().__init__()

        # create the hidden layers
        modules = nn.ModuleList([nn.Flatten()])
        for hidden_layer_size in hidden_layer_sizes:
            modules.append(nn.Linear(input_size, hidden_layer_size))
            modules.append(nn.ReLU())
            input_size = hidden_layer_size

        # define the output layer for the classifier
        modules.append(nn.Linear(input_size, n_classes))

        # create the MLP from the modules
        self.mlp = nn.Sequential(*modules)

    def forward(self, X):
        return self.mlp(X)



In [15]:
_ = torch.manual_seed(42)

# define the model function
fashion_mnist_model_fn = MLPClassifier(
    input_size=28 * 28,
    hidden_layer_sizes=[256, 128],
    n_classes=10
)
fashion_mnist_model_fn = fashion_mnist_model_fn.to(DEVICE)


In [16]:
cross_entropy_loss = nn.CrossEntropyLoss()

sgd = optim.SGD(
    fashion_mnist_model_fn.parameters(),
    lr=1e-1
)

accuracy = (
    torchmetrics.Accuracy(
        task="multiclass",
        num_classes=10,
    ).to(DEVICE)
)


In [17]:
%%timeit -n 1 -r 1

history = train(
    model_fn=fashion_mnist_model_fn,
    criterion=cross_entropy_loss,
    optimizer=sgd,
    metric=accuracy,
    train_data_loader=train_data_loader,
    val_data_loader=val_data_loader,
    n_epochs=20,
    log_epochs=1
)

Epoch 1/20, train loss: 0.6026, val loss: 0.7308, train metric: 0.7790, val metric: 0.7354
Epoch 2/20, train loss: 0.4043, val loss: 0.4243, train metric: 0.8501, val metric: 0.8448
Epoch 3/20, train loss: 0.3592, val loss: 0.3661, train metric: 0.8671, val metric: 0.8652
Epoch 4/20, train loss: 0.3321, val loss: 0.3552, train metric: 0.8759, val metric: 0.8676
Epoch 5/20, train loss: 0.3117, val loss: 0.3538, train metric: 0.8840, val metric: 0.8660
Epoch 6/20, train loss: 0.2959, val loss: 0.3439, train metric: 0.8896, val metric: 0.8774
Epoch 7/20, train loss: 0.2857, val loss: 0.3357, train metric: 0.8930, val metric: 0.8778
Epoch 8/20, train loss: 0.2717, val loss: 0.3276, train metric: 0.8980, val metric: 0.8826
Epoch 9/20, train loss: 0.2612, val loss: 0.3269, train metric: 0.9018, val metric: 0.8824
Epoch 10/20, train loss: 0.2521, val loss: 0.3712, train metric: 0.9037, val metric: 0.8662
Epoch 11/20, train loss: 0.2425, val loss: 0.4073, train metric: 0.9073, val metric: 0.84

## Compile your PyTorch models for better performance!

In [ ]:
torch.compile?

Always make sure your model is moved to the preferred device prior to compiling the model. Why?

* **Compilation is device-specific:** `torch.compile` optimizes the model for a specific runtime environment and hardware. If you compile on the CPU and then move the model to the GPU, the compiled optimizations might not be fully transferable or optimal for the GPU, potentially leading to less efficient execution or even requiring recompilation under the hood.
* **Device-aware optimizations:** Compiling the model when it's already on the target device allows `torch.compile` to leverage device-specific instructions and memory layouts during the optimization process, resulting in more efficient compiled code for that particular device.
* **Avoiding potential issues:** While `torch.compile` is becoming more robust, moving a compiled model to a different device can sometimes lead to unexpected behavior or performance regressions, especially with advanced features like CUDA graphs. Performing the device transfer first simplifies the compilation process and reduces the chances of such issues.

In [26]:
# confirm that model is on the GPU
print(next(fashion_mnist_model_fn.parameters()).device)

cuda:0


In [27]:
compiled_fashion_mnist_model_fn = torch.compile(
    fashion_mnist_model_fn,
    backend="inductor",
    mode="default"
)

In [28]:
%%timeit -n 1 -r 1

history = train(
    model_fn=compiled_fashion_mnist_model_fn,
    criterion=cross_entropy_loss,
    optimizer=sgd,
    metric=accuracy,
    train_data_loader=train_data_loader,
    val_data_loader=val_data_loader,
    n_epochs=20,
    log_epochs=1
)

W1112 07:52:56.157000 798 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1/20, train loss: 0.1794, val loss: 0.3669, train metric: 0.9321, val metric: 0.8782
Epoch 2/20, train loss: 0.1759, val loss: 0.3285, train metric: 0.9311, val metric: 0.8872
Epoch 3/20, train loss: 0.1730, val loss: 0.3458, train metric: 0.9337, val metric: 0.8904
Epoch 4/20, train loss: 0.1658, val loss: 0.3290, train metric: 0.9370, val metric: 0.8924
Epoch 5/20, train loss: 0.1633, val loss: 0.3321, train metric: 0.9375, val metric: 0.8870
Epoch 6/20, train loss: 0.1600, val loss: 0.3444, train metric: 0.9377, val metric: 0.8838
Epoch 7/20, train loss: 0.1559, val loss: 0.3667, train metric: 0.9402, val metric: 0.8812
Epoch 8/20, train loss: 0.1518, val loss: 0.3485, train metric: 0.9412, val metric: 0.8902
Epoch 9/20, train loss: 0.1481, val loss: 0.3473, train metric: 0.9434, val metric: 0.8878
Epoch 10/20, train loss: 0.1422, val loss: 0.3504, train metric: 0.9451, val metric: 0.8876
Epoch 11/20, train loss: 0.1391, val loss: 0.3718, train metric: 0.9462, val metric: 0.88

## Predicting using the trained model

### Predicting class labels

In [ ]:
def predict(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
    class_indices = torch.argmax(y_pred_logits, dim=1)
    return class_indices

In [ ]:
X_new, y_new = next(iter(val_data_loader))

In [ ]:
X_new.device

device(type='cpu')

In [ ]:
X_new = X_new.to(DEVICE)

In [ ]:
class_indices = predict(X_new, fashion_mnist_model_fn)
print(class_indices)

tensor([7, 4, 4, 5, 9, 8, 7, 7, 7, 4, 4, 4, 7, 7, 6, 0, 1, 3, 7, 1, 2, 4, 4, 0, 6, 2, 4, 8, 5, 7, 2, 9],
       device='cuda:0')


In [ ]:
class_labels = [train_val_dataset.classes[i] for i in class_indices]
print(class_labels)

['Sneaker', 'Coat', 'Coat', 'Sandal', 'Ankle boot', 'Bag', 'Sneaker', 'Sneaker', 'Sneaker', 'Coat', 'Coat', 'Coat', 'Sneaker', 'Sneaker', 'Shirt', 'T-shirt/top', 'Trouser', 'Dress', 'Sneaker', 'Trouser', 'Pullover', 'Coat', 'Coat', 'T-shirt/top', 'Shirt', 'Pullover', 'Coat', 'Bag', 'Sandal', 'Sneaker', 'Pullover', 'Ankle boot']


### Predicting class probabilities

In [ ]:
def predict_proba(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        y_pred_proba = torch.softmax(y_pred_logits, dim=1)
    return y_pred_proba


In [ ]:
class_probas = predict_proba(X_new, fashion_mnist_model_fn)
print(class_probas.round(decimals=3))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9630, 0.0000, 0.0370],
        [0.0000, 0.0000, 0.0310, 0.0000, 0.9690, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0160, 0.0000, 0.2220, 0.0040, 0.6900, 0.0000, 0.0490, 0.0000, 0.0190, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0010, 0.0000, 0.9990],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0020, 0.0000, 0.8270, 0.0030, 0.1680],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9490, 0.0000, 0.0510],
        [0.0000, 0.0010, 0.0070, 0.0360, 0.9550, 0.0000, 0.0010, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0010, 0.4240, 0.5660, 0.0000, 0.0090, 0.0000, 0.0000, 0.0000],
        [0

### Top-k predictions

In [ ]:
def predict_topk(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        _, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
    return topk_class_indices


def predict_topk_proba(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        topk_logits, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
        topk_probas = torch.softmax(topk_logits, dim=1)
    return topk_probas



In [ ]:
top3_class_indices = predict_topk(X_new, fashion_mnist_model_fn, k=3)
print(top3_class_indices)

tensor([[7, 9, 5],
        [4, 2, 6],
        [4, 2, 6],
        [5, 0, 9],
        [9, 7, 5],
        [8, 4, 3],
        [7, 9, 5],
        [7, 9, 8],
        [7, 9, 5],
        [4, 3, 2],
        [4, 3, 6],
        [4, 0, 2],
        [7, 5, 3],
        [7, 5, 9],
        [6, 4, 3],
        [0, 6, 2],
        [1, 0, 3],
        [3, 0, 6],
        [7, 9, 5],
        [1, 3, 0],
        [2, 4, 0],
        [4, 2, 6],
        [4, 3, 6],
        [0, 6, 2],
        [6, 4, 2],
        [2, 0, 4],
        [4, 3, 2],
        [8, 0, 9],
        [5, 7, 0],
        [7, 9, 8],
        [2, 6, 4],
        [9, 7, 5]], device='cuda:0')


In [ ]:
top3_class_labels = []
for class_indices in top3_class_indices:
    top3_class_labels.append(
        [train_val_dataset.classes[i] for i in class_indices]
    )
print(top3_class_labels)


In [ ]:
top3_probas = predict_topk_proba(X_new, fashion_mnist_model_fn, k=3)
print(top3_probas.round(decimals=3))